In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# تحميل النموذج والأدوات المحفوظة
# ==========================================

import pickle
from tensorflow.keras.models import load_model

PROJECT_PATH = "/content/drive/MyDrive/Arabic_News_LSTM_Project_Final"


# تحميل النموذج الأفضل
model = load_model(
    f"{PROJECT_PATH}/best_model.keras"
)


# تحميل Tokenizer
with open(
    f"{PROJECT_PATH}/tokenizer.pkl",
    "rb"
) as f:
    tokenizer = pickle.load(f)


# تحميل Label Encoder
with open(
    f"{PROJECT_PATH}/label_encoder.pkl",
    "rb"
) as f:
    label_encoder = pickle.load(f)


# نفس طول الإدخال المستخدم في التدريب
MAX_LEN = 800


print("Model loaded ✅")
print("Tokenizer loaded ✅")
print("Label encoder loaded ✅")
print("MAX_LEN:", MAX_LEN)

Model loaded ✅
Tokenizer loaded ✅
Label encoder loaded ✅
MAX_LEN: 800


In [ ]:
# ==========================================
# دالة تنظيف النصوص العربية
# نفس المستخدمة أثناء التدريب
# ==========================================

import re


def clean_arabic_text(text):

    text = str(text)

    # إزالة الروابط
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # إزالة البريد الإلكتروني
    text = re.sub(r'\S+@\S+', ' ', text)

    # إزالة التشكيل
    arabic_diacritics = re.compile(
        r'[\u0617-\u061A\u064B-\u0652]'
    )

    text = arabic_diacritics.sub('', text)

    # توحيد الحروف
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)

    # الإبقاء على العربي والإنجليزي والأرقام
    text = re.sub(
        r"[^a-zA-Z0-9ء-ي\s]",
        " ",
        text
    )

    # إزالة المسافات الزائدة
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


print("Cleaning function loaded ✅")

Cleaning function loaded ✅


In [ ]:
# ==========================================
# دالة تصنيف خبر جديد
# نفس مسار التدريب بالكامل
# ==========================================

from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np


def predict_news(text):

    # تنظيف النص
    clean_text = clean_arabic_text(text)

    # تحويل الكلمات إلى أرقام
    sequence = tokenizer.texts_to_sequences(
        [clean_text]
    )

    # نفس Padding المستخدم أثناء التدريب
    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN
    )

    # التنبؤ
    prediction = model.predict(
        padded
    )

    # اختيار أعلى احتمال
    predicted_class = np.argmax(
        prediction,
        axis=1
    )[0]

    # تحويل الرقم إلى اسم الفئة
    label = label_encoder.inverse_transform(
        [predicted_class]
    )[0]

    # نسبة الثقة
    confidence = np.max(prediction)

    return label, confidence


print("Prediction function loaded ✅")

Prediction function loaded ✅


In [ ]:
# ==========================================
# تجربة 2: خبر رياضي
# ==========================================

text = """
حقق الفريق الفوز في المباراة النهائية
بعد تسجيل هدف في الدقائق الأخيرة
ليتوج ببطولة الدوري
"""

result = predict_news(text)

print("Predicted class:", result[0])
print("Confidence:", result[1])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Predicted class: Sports
Confidence: 0.9992041


In [ ]:
# ==========================================
# تجربة 1: خبر تقني
# ==========================================

text = """
أعلنت شركة آبل عن إطلاق هاتف ذكي جديد
مزود بمعالج قوي وتقنيات ذكاء اصطناعي حديثة
وكاميرا محسنة ضمن منافسة الشركات التقنية
"""

result = predict_news(text)

print("Predicted class:", result[0])
print("Confidence:", result[1])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step
Predicted class: Tech
Confidence: 0.9999051
